# 🍜 STEP 03 — 멀티 에이전트: AgentGroupChat + 트리아지 오케스트레이터

**STEP 02**에서 만든 3개 전문 에이전트를 하나의 그룹으로 묶고,
트리아지 에이전트(오케스트레이터)가 요청을 적절한 에이전트에 라우팅합니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| `AgentGroupChat` | 여러 에이전트가 참여하는 대화 그룹 |
| `KernelFunctionSelectionStrategy` | 다음 에이전트를 LLM 프롬프트로 선택 |
| `KernelFunctionTerminationStrategy` | 종료 조건을 LLM 프롬프트로 판단 |
| Plugin-as-Agent 패턴 | 에이전트를 다른 에이전트의 플러그인으로 등록 |

---

## F&B 시스템 흐름
```
사용자 입력
    ↓
TriageAgent (오케스트레이터) ← SelectionStrategy가 제어
    ├→ LegalTaxAgent  (법령·세무 질문)
    ├→ LocationAgent  (상권 분석 질문)
    └→ FinanceAgent   (재무 분석 질문)
    ↓
TerminationStrategy가 "완료" 판단 → 최종 응답
```

In [ ]:
import asyncio, os, json
from typing import Annotated
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies import (
    KernelFunctionSelectionStrategy,
    KernelFunctionTerminationStrategy,
)
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.contents import ChatHistoryTruncationReducer
from semantic_kernel.functions import kernel_function, KernelArguments, KernelFunctionFromPrompt
# ✅ [FIX] allow_dangerously_set_content 오류 해결을 위한 임포트 추가
from semantic_kernel.prompt_template import PromptTemplateConfig, InputVariable

load_dotenv()
print('임포트 완료')


In [ ]:
# STEP 02와 동일한 헬퍼 + 플러그인 (Azure RAI 우회: 모든 description/Annotated 영어)
def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        api_key=os.getenv('AZURE_OPENAI_API_KEY'),
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview'),
    ))
    return k

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

class FnbLegalPlugin:
    @kernel_function(description="Returns F&B business license requirements and required documents by business type.")
    def get_license_requirements(self,
        business_type: Annotated[str, "Business type (e.g. general restaurant, cafe)"],
        area_sqm:      Annotated[float, "Business area in square meters"]) -> str:
        docs = {'일반음식점': '영업신고(보건소)·위생교육(6h)·소방완비증명(300㎡↑)',
                '카페':       '영업신고(보건소)·위생교육(3h)'}
        return docs.get(business_type, '해당 정보 없음')

    @kernel_function(description="Returns recommended tax type based on expected annual revenue in KRW.")
    def get_tax_type_guide(self,
        annual_revenue_krw: Annotated[int, "Expected annual revenue in KRW"]) -> str:
        return '간이과세' if annual_revenue_krw < 104_000_000 else '일반과세'

# ✅ [FIX] FnbLocationPlugin: Mock → 서울시 상권분석 API (OA-22175, OA-22172)
# API 문서: https://data.seoul.go.kr/dataList/OA-22175/S/1/datasetView.do
# 사용 API:
#   - 추정매출-행정동 (OA-22175): 행정동별 업종 추정 월매출
#   - 점포-행정동    (OA-22172): 행정동별 업종 점포 수
# 공통 파라미터: SERVICE=SejongSales, Type=json, KEY=발급키
import urllib.request, urllib.parse

# 행정동 코드 매핑 (주요 상권)
DONG_CODE_MAP = {
    '홍대': '1144068000',   # 서울 마포구 서교동
    '홍대입구': '1144068000',
    '강남': '1168064000',   # 서울 강남구 역삼1동
    '역삼': '1168064000',
    '신촌': '1141053000',   # 서울 서대문구 신촌동
    '이태원': '1117064000', # 서울 용산구 이태원1동
    '건대': '1130571000',   # 서울 광진구 화양동
    '합정': '1144062000',   # 서울 마포구 합정동
}

# 업종 코드 매핑 (서울시 상권 서비스 업종 분류)
INDUSTRY_CODE_MAP = {
    '카페': 'CS100001',      # 커피/음료
    '일반음식점': 'CS100002', # 한식/일반음식
    '치킨': 'CS100003',
    '피자': 'CS100004',
    '패스트푸드': 'CS100005',
}

class FnbLocationPlugin:
    @kernel_function(
        description=(
            "Queries Seoul commercial area analysis API (OA-22175, OA-22172) "
            "to retrieve estimated monthly sales and store count for a given location and business type."
        )
    )
    def analyze_commercial_area(
        self,
        location:      Annotated[str, "Location name in Korean (e.g. 홍대, 강남, 신촌)"],
        business_type: Annotated[str, "Business type in Korean (e.g. 카페, 일반음식점)"],
    ) -> str:
        api_key = os.getenv('SEOUL_API_KEY', '')
        dong_code = DONG_CODE_MAP.get(location, '')

        # API 키 또는 행정동 코드 없으면 안내 메시지 반환
        if not api_key or not dong_code:
            missing = []
            if not api_key:   missing.append('SEOUL_API_KEY (.env)')
            if not dong_code: missing.append('지원 지역: ' + ', '.join(DONG_CODE_MAP.keys()))
            return json.dumps({
                'error': 'API 설정 필요',
                'missing': missing,
                'location': location,
                'business_type': business_type,
                'fallback': {
                    'daily_population': 45000,
                    'competitors': 12,
                    'avg_monthly_sales_krw': 18500000,
                },
            }, ensure_ascii=False)

        industry_code = INDUSTRY_CODE_MAP.get(business_type, 'CS100002')
        quarter = '20243'  # 2024년 3분기 (최신)
        result = {}

        try:
            # ── API 1: 추정매출-행정동 (OA-22175) ──
            sales_url = (
                'http://openapi.seoul.go.kr:8088/%s/json/SejongSales/1/5/'
                '?STDR_YY_CD=%s&STDR_QU_CD=%s&ADSTRD_CD=%s&SVC_INDUTY_CD=%s'
            ) % (
                urllib.parse.quote(api_key),
                quarter[:4], quarter[4:],
                dong_code, industry_code,
            )
            with urllib.request.urlopen(sales_url, timeout=5) as r:
                data = json.loads(r.read().decode())
            rows = data.get('SejongSales', {}).get('row', [])
            if rows:
                row = rows[0]
                result['monthly_sales_krw']     = int(row.get('THSMON_SELNG_AMT', 0))
                result['monthly_sales_count']    = int(row.get('THSMON_SELNG_CO', 0))
                result['weekday_sales_rate']     = float(row.get('MON_SELNG_RATE', 0))
                result['weekend_sales_rate']     = float(row.get('SUN_SELNG_RATE', 0))
            else:
                result['monthly_sales_krw'] = None

            # ── API 2: 점포-행정동 (OA-22172) ──
            store_url = (
                'http://openapi.seoul.go.kr:8088/%s/json/SejongStoreW/1/5/'
                '?STDR_YY_CD=%s&STDR_QU_CD=%s&ADSTRD_CD=%s&SVC_INDUTY_CD=%s'
            ) % (
                urllib.parse.quote(api_key),
                quarter[:4], quarter[4:],
                dong_code, industry_code,
            )
            with urllib.request.urlopen(store_url, timeout=5) as r:
                data2 = json.loads(r.read().decode())
            rows2 = data2.get('SejongStoreW', {}).get('row', [])
            if rows2:
                row2 = rows2[0]
                result['store_count']    = int(row2.get('STOR_CO', 0))
                result['open_rate']      = float(row2.get('OPBIZ_RATE', 0))
                result['close_rate']     = float(row2.get('CLSBIZ_RATE', 0))
        except Exception as e:
            result['api_error'] = str(e)
            result['fallback'] = {
                'daily_population': 45000,
                'competitors': 12,
                'avg_monthly_sales_krw': 18500000,
            }

        result['location']      = location
        result['business_type'] = business_type
        result['dong_code']     = dong_code
        result['industry_code'] = industry_code
        result['quarter']       = quarter
        result['source']        = '서울시 상권분석서비스 (OA-22175, OA-22172)'
        return json.dumps(result, ensure_ascii=False)

class FnbFinancePlugin:
    @kernel_function(description="Calculates break-even point using initial investment, fixed costs, avg revenue per customer, and variable cost ratio.")
    def calculate_bep(self,
        initial_investment_krw:       Annotated[int,   "Total initial investment amount in KRW"],
        monthly_fixed_cost_krw:       Annotated[int,   "Monthly fixed costs (rent, labor, etc.) in KRW"],
        avg_revenue_per_customer_krw: Annotated[int,   "Average revenue per customer in KRW"],
        variable_cost_ratio:          Annotated[float, "Variable cost ratio between 0 and 1 (e.g. 0.35 for 35%)"]) -> str:
        cm  = 1 - variable_cost_ratio
        rev = monthly_fixed_cost_krw / cm
        cus = rev / avg_revenue_per_customer_krw
        return 'monthly_bep_revenue: %s KRW, monthly_bep_customers: %d persons' % ('{:,.0f}'.format(rev), int(cus))

print('헬퍼·플러그인 정의 완료')


---
## 1. 에이전트 3종 생성 (STEP 02 복습)

In [ ]:
# ── LegalTaxAgent ──
legal_k = make_kernel()
legal_k.add_plugin(FnbLegalPlugin(), plugin_name='FnbLegal')
legal_tax_agent = ChatCompletionAgent(
    name='LegalTaxAgent',
    description='F&B startup legal, license, and tax specialist agent',
    instructions=(
        'You are an F&B startup legal and tax expert. '
        'Provide accurate information based on Korean Food Sanitation Act, '
        'Commercial Building Lease Protection Act, Labor Standards Act, and VAT Act. '
        'Always cite the relevant legal article. '
        'Always respond in Korean.'
    ),
    kernel=legal_k,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── LocationAgent ──
loc_k = make_kernel()
loc_k.add_plugin(FnbLocationPlugin(), plugin_name='FnbLocation')
location_agent = ChatCompletionAgent(
    name='LocationAgent',
    description='Commercial area analysis specialist using foot traffic, competitor data, and Kakao Maps',
    instructions=(
        'You are an F&B startup commercial area analysis expert. '
        'Provide foot traffic, competitor count, and rent level with concrete numbers. '
        'Structure your analysis as Risk / Opportunity factors. '
        'Always respond in Korean.'
    ),
    kernel=loc_k,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── FinanceAgent ──
fin_k = make_kernel()
fin_k.add_plugin(FnbFinancePlugin(), plugin_name='FnbFinance')
finance_agent = ChatCompletionAgent(
    name='FinanceAgent',
    description='BEP calculation, scenario analysis, and risk simulation specialist',
    instructions=(
        'You are an F&B startup finance analysis expert. '
        'Provide BEP calculation, scenario-based revenue forecast, and risk assessment. '
        'Always include optimistic, base, and pessimistic scenarios. '
        'Always respond in Korean.'
    ),
    kernel=fin_k,
    arguments=KernelArguments(settings=make_auto_settings()),
)

print('에이전트 3종 생성 완료')


---
## 2. SelectionStrategy — 다음 에이전트를 LLM이 선택

> **핵심 개념**: 누가 다음에 말할지를 LLM 프롬프트로 결정합니다.
> 프롬프트가 에이전트 이름만 반환하면 SK가 해당 에이전트를 호출합니다.

In [ ]:
AGENT_NAMES = ['LegalTaxAgent', 'LocationAgent', 'FinanceAgent']

# ⚠️ Azure RAI filter: strategy prompts must be in English
selection_prompt = """
You are a router for an F&B startup multi-agent system.
Analyze the conversation history and return ONLY the name of the next agent to respond.

[Available Agents]
{{$agents}}

[Routing Rules]
- License, sanitation, tax, legal matters         → LegalTaxAgent
- Commercial area, foot traffic, competitors, rent → LocationAgent
- Investment, BEP, revenue forecast, risk          → FinanceAgent
- For complex requests, select the most urgent one first

[Conversation History]
{{$history}}

Return ONLY the agent name (no explanation):
"""

termination_prompt = """
Review the conversation below and determine if the user's question has been fully answered.

[Criteria]
- A specialist agent has provided a concrete answer → terminate
- The answer is incomplete or follow-up is needed  → continue

[Conversation History]
{{$history}}

Return ONLY '완료' (terminate) or '계속' (continue):
"""

strategy_kernel = make_kernel()

# ✅ [FIX] allow_dangerously_set_content=True 추가
# 원인: SK가 {{$history}}, {{$agents}} 변수를 프롬프트에 주입할 때
#        ChatHistory 객체를 자동 인코딩하지 못해서 NotImplementedError 발생
# 해결: PromptTemplateConfig에 InputVariable로 명시적 허용 선언
selection_function = KernelFunctionFromPrompt(
    function_name='agent_selection',
    prompt_template_config=PromptTemplateConfig(
        template=selection_prompt,
        template_format='semantic-kernel',
        input_variables=[
            InputVariable(name='agents',  allow_dangerously_set_content=True),
            InputVariable(name='history', allow_dangerously_set_content=True),
        ],
    ),
)

termination_function = KernelFunctionFromPrompt(
    function_name='termination_check',
    prompt_template_config=PromptTemplateConfig(
        template=termination_prompt,
        template_format='semantic-kernel',
        input_variables=[
            InputVariable(name='history', allow_dangerously_set_content=True),
        ],
    ),
)

print('전략 프롬프트 정의 완료')


---
## 3. AgentGroupChat 구성 — 오케스트레이터 완성

In [ ]:
def create_fnb_group_chat() -> AgentGroupChat:
    """
    F&B 창업 지원 AgentGroupChat을 생성합니다.

    구성:
    - 참여 에이전트: LegalTaxAgent, LocationAgent, FinanceAgent
    - SelectionStrategy: LLM이 다음 에이전트를 선택
    - TerminationStrategy: LLM이 종료 여부를 판단
    - 최대 반복: 10회 (무한 루프 방지)
    """
    # 대화 기록을 너무 길게 보내지 않도록 마지막 5개 메시지만 전략 프롬프트에 포함
    history_reducer = ChatHistoryTruncationReducer(target_count=5)

    selection_strategy = KernelFunctionSelectionStrategy(
        function=selection_function,
        kernel=strategy_kernel,
        # 프롬프트 결과에서 에이전트 이름을 파싱하는 함수
        result_parser=lambda result: str(result.value[0]).strip(),
        history_variable_name="history",
        history_reducer=history_reducer,
    )

    termination_strategy = KernelFunctionTerminationStrategy(
        function=termination_function,
        kernel=strategy_kernel,
        # "완료" 텍스트가 응답에 포함되면 대화 종료
        result_parser=lambda result: "완료" in str(result.value[0]),
        history_variable_name="history",
        history_reducer=history_reducer,
        maximum_iterations=10,  # 안전장치: 최대 10회 반복
    )

    return AgentGroupChat(
        agents=[legal_tax_agent, location_agent, finance_agent],
        selection_strategy=selection_strategy,
        termination_strategy=termination_strategy,
    )

print("AgentGroupChat 팩토리 함수 정의 완료")

---
## 4. 라우팅 테스트 — 각 전문 에이전트로 자동 라우팅

In [ ]:
from semantic_kernel.contents import ChatMessageContent, AuthorRole

async def ask_group(question: str):
    """GroupChat에 질문을 던지고 에이전트 응답을 출력합니다."""
    chat = create_fnb_group_chat()
    await chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content=question)
    )
    print('[질문] %s' % question)
    print('=' * 60)
    async for msg in chat.invoke():
        print('[%s]' % msg.name)
        print(msg.content)
        print('-' * 60)

# 테스트 1: 법령 질문 → LegalTaxAgent 라우팅
# ⚠️ Azure RAI: English question + 'Respond in Korean'
await ask_group(
    'I am planning to open a 30-pyeong general restaurant in Seoul. '
    'Please explain the sanitation license procedures. Respond in Korean.'
)


In [ ]:
# 테스트 2: 상권 질문 → LocationAgent 라우팅
await ask_group(
    'Please analyze the commercial area near Hongik University Station in Mapo-gu, Seoul for a cafe. '
    'Include foot traffic and competitor analysis. Respond in Korean.'
)


In [ ]:
# 테스트 3: 재무 질문 → FinanceAgent 라우팅
await ask_group(
    'Calculate BEP with initial investment 80,000,000 KRW, '
    'monthly fixed cost 3,500,000 KRW, avg revenue per customer 7,000 KRW, '
    'variable cost ratio 35%. Respond in Korean.'
)


In [ ]:
# 테스트 4: 복합 질문 → 순차 라우팅 (LegalTaxAgent → FinanceAgent)
await ask_group(
    'Please review the lease contract and help prepare the food business registration form. '
    'Also calculate BEP with monthly rent 3,000,000 KRW as fixed cost. Respond in Korean.'
)


---
## 5. [심화] Plugin-as-Agent 패턴

> **다른 접근법**: 에이전트를 `GroupChat` 대신 트리아지 에이전트의 **플러그인**으로 등록하는 방법.
> 
> - 장점: 단순함, 트리아지 에이전트가 직접 결과를 합성 가능
> - 단점: 병렬 호출 어려움, 각 에이전트 결과를 별도 스트리밍 불가
> 
> 아키텍처 다이어그램의 `HandoffBuilder` 패턴과 유사합니다.

In [ ]:
class AgentAsPlugin:
    """Adapter that wraps an agent as a plugin function."""

    def __init__(self, agent: ChatCompletionAgent):
        self._agent = agent

    @kernel_function(description="Delegates legal and tax questions to the LegalTax specialist agent.")
    async def ask_legal_tax(
        self,
        question: Annotated[str, "Legal or tax related question in English"],
    ) -> str:
        response = await self._agent.get_response(messages=question)
        return response.content


triage_kernel = make_kernel()
triage_kernel.add_plugin(AgentAsPlugin(legal_tax_agent), plugin_name='SubAgents')

triage_agent = ChatCompletionAgent(
    name='TriageAgent',
    description='Orchestrator that classifies user requests and delegates to specialist agents',
    instructions=(
        'You are an F&B startup support assistant. '
        'For legal and tax questions, call the ask_legal_tax function. '
        'Always respond in Korean.'
    ),
    kernel=triage_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

response = await triage_agent.get_response(
    messages='How many hours of sanitation training are required to open a cafe? Respond in Korean.'
)
print('[TriageAgent (Plugin-as-Agent) 응답]')
print(response.content)


---
## 6. ✅ STEP 03 정리

| 배운 것 | 코드 패턴 | 다음 단계 연결 |
|---------|-----------|---------------|
| AgentGroupChat | `AgentGroupChat(agents=[...])` | STEP 04 MCP 툴 연동 |
| SelectionStrategy | `KernelFunctionSelectionStrategy` | 라우팅 로직의 핵심 |
| TerminationStrategy | `KernelFunctionTerminationStrategy` | 종료 조건 제어 |
| Plugin-as-Agent | 에이전트 → 플러그인 래핑 | HandoffBuilder 대안 패턴 |

---

### 다음 STEP 미리보기
```
STEP 04: MCP (Model Context Protocol) 연동
  - 카카오 지도 API를 MCP 서버로 연동
  - 국가법령정보센터 API를 MCP 툴로 등록
  - SK에서 MCP 서버 호출 패턴
```